In this file we will build the recommendation system.

In [ ]:
# Importing the libraries
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Loading the cleaned dataframe
df_clean = pd.read_csv('../clean_data/df_clean.csv')

# To prevent truncating
pd.set_option('display.max_columns', None)

# Checking the sample rows of this dataframe
df_clean.sample(7)

,TransactionId,UserId,VisitYear,VisitMonth,VisitModeId,AttractionId,Rating,CityId,CityName,CountryId,Country,RegionId,Region,ContinentId,Continent,AttractionCityId,AttractionTypeId,Attraction,AttractionType,VisitMode,Month
18439,27837,25040,2014,9,3,841,4,1434,Melbourne,51,United States,8,Northern America,2,America,1,92,Waterbom Bali,Water Parks,Family,September
50941,207952,30333,2017,8,4,1278,5,3551,Singapore,106,Singapore,14,South East Asia,3,Asia,3,45,Ullen Sentalu Museum,History Museums,Friends,August
42598,144241,10652,2018,1,3,369,2,3551,Singapore,106,Singapore,14,South East Asia,3,Asia,1,13,Kuta Beach - Bali,Beaches,Family,January
50913,207811,21128,2016,1,4,1278,4,3027,Batam,101,Indonesia,14,South East Asia,3,Asia,3,45,Ullen Sentalu Museum,History Museums,Friends,January
39163,104927,35344,2018,5,2,737,4,337,Cambridge,48,Canada,8,Northern America,2,America,1,76,Tanah Lot Temple,Religious Sites,Couples,May
33166,69350,55260,2018,8,3,748,3,7555,Valbonne,159,France,21,Western Europe,5,Europe,1,72,Tegalalang Rice Terrace,Points of Interest & Landmarks,Family,August
34108,70643,58237,2017,4,4,748,5,493,Regina,48,Canada,8,Northern America,2,America,1,72,Tegalalang Rice Terrace,Points of Interest & Landmarks,Friends,April


In [3]:
# Creating the dataframe for the recommendation system
df_rec = df_clean[['UserId', 'Attraction', 'Rating']].copy()

# Checking the sample rows of this dataframe
df_rec.sample(7)

,UserId,Attraction,Rating
25855,87163,Sanur Beach,5
7997,5737,Sacred Monkey Forest Sanctuary,4
40339,59957,Tanah Lot Temple,5
34039,54831,Tegalalang Rice Terrace,5
52020,68457,Water Castle (Tamansari),5
35225,42768,Tegalalang Rice Terrace,3
22804,29050,Nusa Dua Beach,5


In [4]:
# Creating the user-item matrix
user_attr_matrix = df_rec.pivot_table(index='UserId', columns='Attraction', values='Rating')

# Checking the matrix
user_attr_matrix.sample(7)

Attraction,Balekambang Beach,Bromo Tengger Semeru National Park,Coban Rondo Waterfall,Goa Cina Beach,Jodipan Colorful Village,Jomblang Cave,Kalibiru National Park,Khayangan Reflexology & Massage,Kuta Beach - Bali,Malang City Square,Malioboro Road,Merapi Volcano,Mount Semeru Volcano,Museum Malang Tempo Doeloe,Nusa Dua Beach,Ramayana Ballet at Prambanan,Ratu Boko Temple,Sacred Monkey Forest Sanctuary,Sanur Beach,Seminyak Beach,Sempu Island,Sewu Temple,Tanah Lot Temple,Tegalalang Rice Terrace,Tegenungan Waterfall,Ullen Sentalu Museum,Uluwatu Temple,Water Castle (Tamansari),Waterbom Bali,Yogyakarta Palace
UserId,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
29438,NaN,NaN,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,NaN,NaN,5.0,NaN,NaN,NaN
47270,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
71658,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,NaN
2395,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
70939,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,NaN,NaN
54062,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
43673,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# Calculating the number of users
num_users = user_attr_matrix.shape[0]

# Calculating the number of attractions
num_items = user_attr_matrix.shape[1]

# Calculating total possible interactions
total_possible = num_users * num_items

# Calculating actual observed ratings
actual_interactions = user_attr_matrix.count().sum()

# Sparsity calculation
sparsity = 1 - (actual_interactions / total_possible)

print("Number of Users:", num_users)
print("Number of Attractions:", num_items)
print("Total Possible Interactions:", total_possible)
print("Actual Interactions:", actual_interactions)
print("Sparsity:", round(sparsity * 100, 2), "%")

Number of Users: 33507
Number of Attractions: 30
Total Possible Interactions: 1005210
Actual Interactions: 45240
Sparsity: 95.5 %


- Now that we have calculated the sparsity, we see that the data that we have is very sparse (>95%). So, we will approach the item-based collaborative filtering, as very few actual interactions exist as compared to the total possible interactions.

In [6]:
# Filling the missing the values with 0 for the user_attr_matrix
user_attr_filled = user_attr_matrix.fillna(0)

In [7]:
# Computing item-item similarity using cosine similarity
from sklearn.metrics.pairwise import cosine_similarity
attr_similarity = cosine_similarity(user_attr_filled.T)

# Converting the similarity matrix to a DataFrame for better readability
attr_similarity_df = pd.DataFrame(attr_similarity, index=user_attr_matrix.columns, columns=user_attr_matrix.columns)

# Checking the item similarity matrix shape
attr_similarity_df.shape

(30, 30)

In [8]:
# Checking the sample rows of the item similarity matrix
attr_similarity_df.iloc[:5, :5]

Attraction,Balekambang Beach,Bromo Tengger Semeru National Park,Coban Rondo Waterfall,Goa Cina Beach,Jodipan Colorful Village
Attraction,,,,,
Balekambang Beach,1.000000,0.042778,0.126176,0.165588,0.000000
Bromo Tengger Semeru National Park,0.042778,1.000000,0.098347,0.047701,0.071389
Coban Rondo Waterfall,0.126176,0.098347,1.000000,0.126390,0.077877
Goa Cina Beach,0.165588,0.047701,0.126390,1.000000,0.000000
Jodipan Colorful Village,0.000000,0.071389,0.077877,0.000000,1.000000


In [ ]:
# Defining a function that recommends attraction similar to one attraction
def recommend_similar_attractions(attraction, attr_similarity_df, top_n=5):
    if attraction not in attr_similarity_df.columns:
        return f"{attraction} not found"
    
    # Get the similarity scores for the given attraction
    similarity_scores = attr_similarity_df[attraction]

    # Sort the scores in descending order
    similar_attr = similarity_scores.sort_values(ascending=False)[1:top_n+1]

    # Return top_n similar attractions (excluding the attraction itself)
    return similar_attr.index.tolist()

In [20]:
# Example usage
recommend_similar_attractions("Coban Rondo Waterfall")

['Malang City Square',
 'Goa Cina Beach',
 'Balekambang Beach',
 'Bromo Tengger Semeru National Park',
 'Jodipan Colorful Village']

In [21]:
# Saving the attr_similarity_df dataframe
import joblib
joblib.dump(attr_similarity_df, '../artifacts/attr_similarity_df.pkl')
joblib.dump(attr_similarity_df.index.to_list(), '../artifacts/attr_list.pkl')

['../artifacts/attr_list.pkl']